# A* 搜索算法实验：8 数码问题与随机迷宫问题（学生练习版）

本 Notebook 用于学习、实现和考查 A* 搜索算法。程序中已经保留了问题描述、数据结构、部分辅助函数和可视化代码，但删除了若干核心函数的实现，需要学生根据注释和步骤自行完成。

内容分为三部分：

1. A* 搜索算法背景知识
2. A* 搜索求解 8 数码问题
3. A* 搜索求解随机迷宫问题

练习要求：从上到下阅读 Notebook，先理解每个函数的输入输出，再补全标有 `TODO` 的函数。补全后重新运行对应示例单元，观察搜索结果。

## 1. A* 搜索算法背景知识

A* 搜索是一种经典的启发式搜索算法，常用于路径规划、游戏 AI、机器人导航、状态空间搜索等问题。它综合考虑：

- 已经付出的代价：`g(n)`，表示从初始状态到当前状态 `n` 的实际代价。
- 对未来代价的估计：`h(n)`，表示从当前状态 `n` 到目标状态的估计代价，也称为启发函数。
- 综合评价函数：`f(n) = g(n) + h(n)`。

A* 每次从待扩展节点中选择 `f(n)` 最小的节点继续搜索。直观理解是：既希望已经走过的路短，也希望距离目标还近。

### 启发函数的性质

一个好的启发函数能够显著减少搜索空间。常见性质包括：

- 可采纳性：`h(n)` 不高估从 `n` 到目标的真实最小代价。若启发函数可采纳，A* 通常可以保证找到最优解。
- 一致性：对任意相邻节点 `n` 和 `n'`，满足 `h(n) <= cost(n, n') + h(n')`。一致性比可采纳性更强，能让搜索过程更稳定。

### A* 基本步骤

1. 将初始状态放入优先队列 Open List。
2. 从 Open List 中取出 `f` 值最小的状态。
3. 如果该状态是目标状态，则回溯得到解路径。
4. 否则扩展它的后继状态，计算每个后继的 `g`、`h`、`f`。
5. 若后继状态还未访问，或找到了更低代价路径，则更新并加入 Open List。
6. 重复上述过程，直到找到目标或 Open List 为空。

In [ ]:
from heapq import heappop, heappush
from itertools import count


def astar(start, is_goal, neighbors, heuristic):
    """通用 A* 搜索函数，需要学生补全。

    这个函数是两个实验共用的核心算法。8 数码问题和迷宫问题虽然状态表示不同，
    但都可以抽象为：从 start 状态出发，不断扩展后继状态，直到找到目标状态。

    参数：
    - start: 初始状态，必须是可哈希对象，例如 tuple。
    - is_goal(state): 判断 state 是否为目标状态的函数。
    - neighbors(state): 生成后继状态的函数，每个后继写成 (next_state, step_cost)。
    - heuristic(state): 启发函数，估计 state 到目标状态的剩余代价。

    返回：
    - path: 从初始状态到目标状态的状态序列；找不到时返回 None。
    - cost: 最短路径总代价；找不到时返回 None。
    - expanded: 已扩展节点数量，用于评价搜索效率。

    实现提示：
    1. 使用优先队列 open_heap 保存待扩展节点，队列元素可写为：
       (f_score, g_score, tie_id, state)。
    2. 使用 g_score 字典保存从 start 到每个状态的当前最小实际代价。
    3. 使用 came_from 字典保存每个状态的父状态，方便找到目标后回溯路径。
    4. 每次从优先队列中取出 f = g + h 最小的状态。
    5. 若该状态是目标，则从 came_from 中回溯并返回完整路径。
    6. 否则扩展它的后继状态；如果找到更小的 g 值，则更新并重新入队。
    7. 使用 closed 集合记录已经扩展过的状态，避免重复扩展。
    """
    # TODO 1: 初始化 open_heap、tie_breaker、came_from、g_score、closed 和 expanded。
    # 提示：tie_breaker = count() 可以避免两个节点 f 和 g 都相同时 heap 比较 state。

    # TODO 2: 将初始状态加入优先队列。
    # 初始状态的 g 值为 0，f 值为 heuristic(start)。

    # TODO 3: 编写 while open_heap 循环。
    # 每轮取出 f 值最小的 current 状态。

    # TODO 4: 如果 current 已经在 closed 中，说明它是旧队列项，可以跳过。

    # TODO 5: 如果 current 是目标状态，利用 came_from 回溯生成 path 并返回。

    # TODO 6: 将 current 加入 closed，并累计 expanded。

    # TODO 7: 遍历 neighbors(current)，对每个后继计算 tentative_g。
    # 如果 tentative_g 比已知 g_score 更小，则更新 came_from 和 g_score，并把后继加入 open_heap。

    # TODO 8: 如果 open_heap 为空仍未找到目标，返回 None, None, expanded。
    raise NotImplementedError("请补全通用 A* 搜索函数 astar。")

## 2. A* 搜索求解 8 数码问题

8 数码问题也称 8-puzzle。它包含一个 `3 x 3` 棋盘，其中有 8 个数字块和 1 个空格。每一步可以将空格与上、下、左、右相邻的数字块交换，目标是把初始状态移动到目标状态。

本实验中使用：

- `0` 表示空格。
- 状态用长度为 9 的元组表示，例如 `(1, 2, 3, 4, 0, 6, 7, 5, 8)`。
- 启发函数使用曼哈顿距离：每个数字当前位置到目标位置的行列距离之和。

### 8 数码问题求解步骤

1. 确定状态表示：用长度为 9 的 tuple 表示棋盘，按行展开。
2. 确定目标状态：本实验目标为 `(1, 2, 3, 4, 5, 6, 7, 8, 0)`。
3. 判断问题是否有解：比较初始状态和目标状态的逆序数奇偶性。
4. 设计启发函数：实现 `manhattan_8puzzle(state)`，计算所有数字块到目标位置的曼哈顿距离之和。
5. 设计状态扩展函数：实现 `neighbors_8puzzle(state)`，让空格与上下左右合法位置交换，生成新状态。
6. 调用通用 `astar`：传入初始状态、目标判断函数、后继函数和启发函数。
7. 输出结果：打印最少移动步数、扩展节点数和完整解路径。

学生需要重点完成：`astar`、`manhattan_8puzzle`、`neighbors_8puzzle`。

In [ ]:
GOAL_8PUZZLE = (1, 2, 3,
                4, 5, 6,
                7, 8, 0)


def print_board(state):
    """打印 8 数码棋盘。

    参数：
    - state: 长度为 9 的 tuple，例如 (1, 2, 3, 4, 0, 6, 7, 5, 8)。

    实现说明：
    - 每 3 个元素打印为一行。
    - 数字 0 表示空格，为了显示更直观，这里打印为 "_"。
    """
    for i in range(0, 9, 3):
        row = state[i:i + 3]
        print(" ".join("_" if x == 0 else str(x) for x in row))
    print()


def inversion_count(state):
    """计算 8 数码状态的逆序数。

    参数：
    - state: 8 数码状态。

    返回：
    - count_inv: 逆序数。

    实现说明：
    - 逆序是指在状态序列中，前面的数字比后面的数字大。
    - 空格 0 不参与逆序数计算。
    - 例如序列 [2, 1, 3] 中，2 和 1 构成一个逆序。
    """
    nums = [x for x in state if x != 0]
    count_inv = 0
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] > nums[j]:
                count_inv += 1
    return count_inv


def is_solvable_8puzzle(start, goal=GOAL_8PUZZLE):
    """判断 3x3 的 8 数码问题是否有解。

    参数：
    - start: 初始状态。
    - goal: 目标状态。

    返回：
    - True: 有解。
    - False: 无解。

    原理说明：
    - 对于 3x3 棋盘，初始状态与目标状态的逆序数奇偶性相同，问题才有解。
    """
    return inversion_count(start) % 2 == inversion_count(goal) % 2


# 预先记录每个数字在目标状态中的坐标，便于计算曼哈顿距离。
# divmod(index, 3) 会把一维下标转换为二维坐标，例如 index=5 -> (1, 2)。
goal_positions = {value: divmod(index, 3) for index, value in enumerate(GOAL_8PUZZLE)}


def manhattan_8puzzle(state):
    """计算 8 数码状态到目标状态的曼哈顿距离，需要学生补全。

    参数：
    - state: 当前 8 数码状态。

    返回：
    - distance: 所有非空数字块到目标位置的曼哈顿距离之和。

    实现步骤：
    1. 初始化 distance = 0。
    2. 遍历 state 中每个位置 index 和数字 value。
    3. 如果 value == 0，说明是空格，跳过。
    4. 用 divmod(index, 3) 得到 value 当前所在的行 row 和列 col。
    5. 从 goal_positions 中取出 value 的目标行 goal_row 和目标列 goal_col。
    6. 累加 abs(row - goal_row) + abs(col - goal_col)。
    7. 返回 distance。
    """
    # TODO: 按照上面的步骤实现曼哈顿距离启发函数。
    raise NotImplementedError("请补全 8 数码的曼哈顿距离函数 manhattan_8puzzle。")


def neighbors_8puzzle(state):
    """生成 8 数码状态的所有合法后继状态，需要学生补全。

    参数：
    - state: 当前 8 数码状态。

    生成：
    - yield (new_state, 1)，其中 new_state 是移动一步后的新状态，1 是移动代价。

    实现步骤：
    1. 找到空格 0 的位置 zero_index。
    2. 用 divmod(zero_index, 3) 得到空格当前行列坐标。
    3. 枚举四个移动方向：上、下、左、右。
    4. 对每个方向计算新坐标 nr, nc。
    5. 判断新坐标是否仍在 3x3 棋盘内。
    6. 若合法，将空格与目标位置数字交换，得到 new_state。
    7. 使用 yield tuple(new_state), 1 返回后继状态和单步代价。
    """
    # TODO: 按照上面的步骤生成所有合法后继状态。
    raise NotImplementedError("请补全 8 数码后继状态生成函数 neighbors_8puzzle。")

In [ ]:
# 示例：A* 求解 8 数码问题
# 当 astar、manhattan_8puzzle、neighbors_8puzzle 补全后，本单元会输出完整解路径。
start_8puzzle = (1, 2, 3,
                 4, 0, 6,
                 7, 5, 8)

print("初始状态：")
print_board(start_8puzzle)
print("目标状态：")
print_board(GOAL_8PUZZLE)

if not is_solvable_8puzzle(start_8puzzle):
    print("该初始状态无解，请更换初始状态。")
else:
    try:
        path, cost, expanded = astar(
            start=start_8puzzle,
            is_goal=lambda s: s == GOAL_8PUZZLE,
            neighbors=neighbors_8puzzle,
            heuristic=manhattan_8puzzle,
        )

        print(f"最少移动步数：{cost}")
        print(f"扩展节点数：{expanded}")
        print(f"解路径长度：{len(path)} 个状态")
        print("")
        print("解路径：")
        for step, state in enumerate(path):
            print(f"Step {step}")
            print_board(state)
    except NotImplementedError as error:
        print("函数尚未完成：", error)

## 3. A* 搜索求解随机迷宫问题

迷宫问题可以看作网格图上的路径搜索问题。本节随机生成一个一定存在通路的迷宫，并用图片方式展示搜索过程和最终路径。

本节约定：

- 白色格子表示可通行区域。
- 黑色格子表示墙壁。
- 绿色格子表示起点。
- 红色格子表示终点。
- 黄色格子表示 A* 搜索过程中已经扩展过的节点。
- 蓝色格子表示最终找到的最短路径。

随机迷宫使用深度优先搜索回溯法生成。该方法会在网格中挖出通道，因此起点和终点之间天然存在路径。A* 的启发函数仍使用适合四方向移动的曼哈顿距离。

### 随机迷宫问题求解步骤

1. 随机生成迷宫：`generate_random_maze` 已给出，用 `0` 表示通路，用 `1` 表示墙壁。
2. 确定起点和终点：本实验默认起点为左上方通路 `(1, 1)`，终点为右下方通路 `(rows - 2, cols - 2)`。
3. 设计迷宫后继函数：实现 `make_maze_neighbors(maze)`，对当前位置生成上下左右可通行邻居。
4. 设计启发函数：实现 `manhattan_grid(goal)`，返回一个用于估计当前位置到终点距离的函数。
5. 实现带搜索历史的 A*：完成 `astar_with_history`，它与通用 `astar` 类似，但需要记录节点扩展顺序。
6. 展示搜索结果：用黄色显示扩展节点，用蓝色显示最终路径。
7. 展示搜索动画：通过 `animate_astar` 动态观察 A* 的搜索过程。

学生需要重点完成：`make_maze_neighbors`、`manhattan_grid`、`astar_with_history`。

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display


WALL = 1
ROAD = 0


def generate_random_maze(rows=21, cols=31, seed=7):
    """使用深度优先搜索回溯法生成随机迷宫。
    参数：
    - rows: 迷宫行数。若传入偶数，会自动加 1 调整为奇数。
    - cols: 迷宫列数。若传入偶数，会自动加 1 调整为奇数。
    - seed: 随机种子。修改 seed 可以得到不同形状的迷宫。

    返回：
    - maze: numpy 二维数组，0 表示通路，1 表示墙壁。
    - start: 起点坐标。
    - goal: 终点坐标。

    实现说明：
    - 初始时整个网格都是墙壁。
    - 从起点开始，每次随机选择一个尚未访问的方向，把中间墙壁打通。
    - 当当前位置没有可访问方向时回退，这就是 DFS 回溯法。
    - 由于迷宫是从起点逐步挖出来的，所有通路都与起点连通。
    """
    if rows % 2 == 0:
        rows += 1
    if cols % 2 == 0:
        cols += 1

    rng = random.Random(seed)
    maze = np.ones((rows, cols), dtype=int)
    start = (1, 1)
    maze[start] = ROAD
    stack = [start]
    directions = [(-2, 0), (2, 0), (0, -2), (0, 2)]

    while stack:
        r, c = stack[-1]
        candidates = []
        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            if 1 <= nr < rows - 1 and 1 <= nc < cols - 1 and maze[nr, nc] == WALL:
                candidates.append((nr, nc, dr, dc))

        if candidates:
            nr, nc, dr, dc = rng.choice(candidates)
            maze[r + dr // 2, c + dc // 2] = ROAD
            maze[nr, nc] = ROAD
            stack.append((nr, nc))
        else:
            stack.pop()

    goal = (rows - 2, cols - 2)
    return maze, start, goal


def make_maze_neighbors(maze):
    """根据迷宫生成后继状态函数，需要学生补全。

    参数：
    - maze: numpy 二维数组，0 表示通路，1 表示墙壁。

    返回：
    - neighbors(pos): 一个内部函数。输入当前位置 pos=(r, c)，生成所有可通行邻居。

    neighbors(pos) 的实现步骤：
    1. 获取迷宫行数 rows 和列数 cols。
    2. 枚举四个方向：上、下、左、右。
    3. 对当前位置 pos=(r, c)，计算邻居坐标 nr, nc。
    4. 判断邻居是否在迷宫边界内。
    5. 判断 maze[nr, nc] 是否为 ROAD。
    6. 若可以通行，yield (nr, nc), 1，其中 1 表示移动一步的代价。
    """
    # TODO: 在这里定义并返回内部函数 neighbors(pos)。


def manhattan_grid(goal):
    """创建迷宫问题的曼哈顿距离启发函数，需要学生补全。

    参数：
    - goal: 终点坐标，例如 (19, 29)。

    返回：
    - heuristic(pos): 一个函数，输入当前位置 pos，返回 pos 到 goal 的曼哈顿距离。

    实现步骤：
    1. 在 manhattan_grid 内部定义 heuristic(pos)。
    2. pos 和 goal 都是 (row, col) 形式的坐标。
    3. 返回 abs(pos[0] - goal[0]) + abs(pos[1] - goal[1])。
    4. 最后返回 heuristic 函数本身。
    """
    # TODO: 定义并返回 heuristic(pos)。


def astar_with_history(start, is_goal, neighbors, heuristic):
    """带搜索历史记录的 A*，需要学生补全。

    这个函数用于迷宫可视化。它和通用 astar 的核心逻辑相同，区别是：
    - 每次真正扩展一个节点时，都要把该节点加入 expanded_order。
    - 返回结果中包含 expanded_order，供后续动画展示搜索过程。

    参数：
    - start: 起点坐标。
    - is_goal(pos): 判断 pos 是否为终点。
    - neighbors(pos): 生成可通行邻居。
    - heuristic(pos): 估计 pos 到终点的距离。

    返回：
    - path: 从起点到终点的最短路径；找不到时为 None。
    - cost: 最短路径长度；找不到时为 None。
    - expanded_order: A* 实际扩展节点的顺序列表。

    实现步骤：
    1. 初始化 open_heap、tie_breaker、came_from、g_score、closed、expanded_order。
    2. 将起点加入优先队列，起点 g=0，f=heuristic(start)。
    3. 循环取出 f 最小的 current。
    4. 若 current 已扩展过，则跳过。
    5. 将 current 加入 closed 和 expanded_order。
    6. 若 current 是终点，回溯路径并返回 path、cost、expanded_order。
    7. 遍历 current 的邻居，更新更短路径，并将邻居加入优先队列。
    8. 若搜索失败，返回 None、None、expanded_order。
    """
    # TODO: 参考通用 astar 的实现，补全带 expanded_order 的 A* 搜索。


maze_cmap = ListedColormap([
    "white",      # 0: road
    "black",      # 1: wall
    "#f7d154",    # 2: expanded nodes
    "#2f80ed",    # 3: final path
    "#27ae60",    # 4: start
    "#eb5757",    # 5: goal
])


def maze_image(maze, start, goal, expanded=None, path=None):
    """把迷宫、扩展节点和最终路径转换为可绘图的数字矩阵。

    参数：
    - maze: 原始迷宫，0 为通路，1 为墙壁。
    - start: 起点坐标。
    - goal: 终点坐标。
    - expanded: A* 已扩展节点列表，可为 None。
    - path: A* 找到的最终路径，可为 None。

    返回：
    - image: 与 maze 大小相同的矩阵，不同数字对应不同颜色。
    """
    image = maze.astype(int).copy()
    if expanded:
        for r, c in expanded:
            if (r, c) not in (start, goal) and image[r, c] == ROAD:
                image[r, c] = 2
    if path:
        for r, c in path:
            if (r, c) not in (start, goal):
                image[r, c] = 3
    image[start] = 4
    image[goal] = 5
    return image


def show_maze(maze, start, goal, expanded=None, path=None, title="Maze"):
    """以图片方式显示迷宫。
    参数：
    - expanded 不为空时，会用黄色显示 A* 扩展过的节点。
    - path 不为空时，会用蓝色显示最终最短路径。
    """
    image = maze_image(maze, start, goal, expanded, path)
    plt.figure(figsize=(10, 7))
    plt.imshow(image, cmap=maze_cmap, vmin=0, vmax=5)
    plt.title(title)
    plt.xticks([])
    plt.yticks([])
    plt.show()


def animate_astar(maze, start, goal, expanded_order, path, step=3, interval=80):
    """生成 A* 搜索动画。

    参数：
    - expanded_order: A* 节点扩展顺序。
    - path: 最终路径。
    - step: 每帧增加展示的扩展节点数量。step 越大，动画越快。
    - interval: 每帧间隔，单位为毫秒。

    返回：
    - HTML 动画对象，可直接在 Notebook 中显示。
    """
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.set_xticks([])
    ax.set_yticks([])
    frames = list(range(0, len(expanded_order) + 1, step))
    if frames[-1] != len(expanded_order):
        frames.append(len(expanded_order))

    image = ax.imshow(maze_image(maze, start, goal), cmap=maze_cmap, vmin=0, vmax=5)

    def update(frame):
        show_path = path if frame == len(expanded_order) else None
        image.set_data(maze_image(maze, start, goal, expanded_order[:frame], show_path))
        ax.set_title(f"A* Search Process: Expanded {frame} / {len(expanded_order)} Nodes")
        return [image]

    animation = FuncAnimation(fig, update, frames=frames, interval=interval, blit=True)
    plt.close(fig)
    return HTML(animation.to_jshtml())

In [ ]:
# 示例：A* 求解随机迷宫问题
# 当 make_maze_neighbors、manhattan_grid、astar_with_history 补全后，本单元会展示搜索结果和动画。
# 修改 rows、cols 和 seed 可以得到不同规模、不同形状的随机迷宫。
maze, start_maze, goal_maze = generate_random_maze(rows=21, cols=31, seed=12)

show_maze(maze, start_maze, goal_maze, title="Randomly Generated Maze")

try:
    path, cost, expanded_order = astar_with_history(
        start=start_maze,
        is_goal=lambda pos: pos == goal_maze,
        neighbors=make_maze_neighbors(maze),
        heuristic=manhattan_grid(goal_maze),
    )

    print("搜索结果：")
    if path is None:
        print("未找到从起点到终点的路径。")
    else:
        print(f"最短路径长度：{cost}")
        print(f"扩展节点数：{len(expanded_order)}")
        print(f"路径节点数：{len(path)}")
        show_maze(
            maze,
            start_maze,
            goal_maze,
            expanded=expanded_order,
            path=path,
            title="A* Search Result: Yellow = Expanded Nodes, Blue = Shortest Path",
        )

        display(animate_astar(maze, start_maze, goal_maze, expanded_order, path, step=4, interval=80))
except NotImplementedError as error:
    print("函数尚未完成：", error)